# Synthetic weekly sales for the changeover demo
**Demo data only, not real sales.** Generates two years of weekly units and revenue per GTIN and store, so the app and the sales-impact analysis can be shown without real sell-out data.

| Product role | What sells |
| --- | --- |
| `published_future` | the 6 published NIVEA/Henkel changeovers (switch in 2027): old GTIN only |
| `sample_document` | the 4 Unilever sample products: old GTIN (Knorr keeps its GTIN) |
| `past_synthetic` | 6 made-up changeovers in 2026: old GTIN → 3-week overlap → new GTIN with a **planted effect** |
| `control` | 3 never-changing products per category, for before/after comparisons |

**Writes**
* `silver.weekly_sales` (overwrite)
* `silver.synthetic_changeover_truth` (overwrite): the answer key with planted effects; use it only to check the analysis
* `silver.product_launches`: the 6 past changeovers (old → new GTIN), inserted once via MERGE, labelled `synthetic://seed/changeovers/...`

**Deterministic:** each product draws its randomness from its own seed (its old GTIN), so adding or removing products leaves the others unchanged, and `end_week` is fixed so reruns give identical data. Remove later with `DROP TABLE` on the two tables and `DELETE FROM silver.product_launches WHERE source_file LIKE 'synthetic://seed/changeovers/%'`.

In [0]:
import hashlib
import numpy as np, pandas as pd

dbutils.widgets.text("catalog", "patricia_aws_serverless_catalog")
dbutils.widgets.text("end_week", "2026-11-09")  # Monday of the last week of sales; fixed for reproducibility
CATALOG = dbutils.widgets.get("catalog")
END_WEEK = pd.Timestamp(dbutils.widgets.get("end_week"))
assert END_WEEK.weekday() == 0, "end_week must be a Monday"

WEEKS = pd.date_range(END_WEEK - pd.Timedelta(weeks=103), END_WEEK, freq="W-MON")  # 104 Monday-weeks
STORES = {"S01 Berlin": 1.0, "S02 Hamburg": 0.8, "S03 Muenchen": 1.2, "S04 Koeln": 0.7}
SEASON = {  # category: (amplitude, peak week of year)
    "Hautpflege": (0.18, 2), "Sonnenschutz": (0.60, 27), "Rasur": (0.05, 26), "Laundry": (0.04, 1), "Dishwashing": (0.04, 1),
    "Hair care": (0.08, 30), "Body care": (0.15, 28), "Deodorant": (0.25, 29), "Food": (0.20, 3), "Oral care": (0.03, 1)}
TRANSITION_OLD, TRANSITION_NEW = [0.6, 0.3, 0.1], [0.3, 0.6, 0.85]  # share of normal sales per GTIN in the 3 changeover weeks
# Week-to-week noise: negative binomial dispersion (higher = less noise). 200 gives about ±10% per store-week at these volumes;
# product promotions (+30% units, -20% price, flagged in on_promo) are kept, so the analysis should adjust for them.
NB_DISPERSION = 200

def gtin13(body12):
    s = sum(int(d) * (3 if i % 2 == 0 else 1) for i, d in enumerate(reversed(body12)))
    return body12 + str((10 - s % 10) % 10)

def stable(*parts):  # deterministic hash (Python's hash() changes between runs)
    return int(hashlib.md5("|".join(map(str, parts)).encode()).hexdigest(), 16)

print("weeks", WEEKS.min().date(), "to", WEEKS.max().date(), f"({len(WEEKS)})")

In [0]:
def product(client_id, desc, category, size, gtin_old, gtin_new, changeover, effect, role):
    return dict(client_id=client_id, desc=desc, category=category, size=size, gtin_old=gtin_old, gtin_new=gtin_new,
                changeover=pd.Timestamp(changeover) if changeover else None, effect=effect, role=role)

P = []
# Published real changeovers (cutover in 2027: only the old GTIN sells today)
for c, d, cat, size, go, gn, lw in [
    ("NIVEA", "NIVEA Creme Dose 150ml", "Hautpflege", "150 ml", "4005800120015", "4006381335010", "2027-02-15"),
    ("NIVEA", "NIVEA Soft Tube 100ml", "Hautpflege", "100 ml", "4005800120022", "4006381335027", "2027-02-15"),
    ("NIVEA", "NIVEA Men Sensitive Gel 200ml", "Rasur", "200 ml", "4005800120039", "4006381335034", "2027-02-15"),
    ("Henkel", "Persil Universal Gel 1.5L", "Laundry", "1.5 L", "4015000100014", "4015800300010", "2027-03-08"),
    ("Henkel", "Pril Original Dish Soap 500ml", "Dishwashing", "500 ml", "4015000100021", "4015800300027", "2027-03-08"),
    ("Henkel", "Schwarzkopf Gliss Shampoo 250ml", "Hair care", "250 ml", "4015000100038", "4015800300034", "2027-03-08")]:
    P.append(product(c, d, cat, size, go, gn, lw, None, "published_future"))
# Unilever sample document (not published yet); Knorr keeps its GTIN
for d, cat, size, go, gn in [
    ("Dove Deeply Nourishing Body Wash 450ml", "Body care", "450 ml", "8710400100010", "8712000300012"),
    ("Rexona Men Invisible Roll-on 50ml", "Deodorant", "50 ml", "8710400100027", "8712000300029"),
    ("Knorr Vegetable Bouillon 8x10g", "Food", "80 g", "8710400100034", "8710400100034"),
    ("Signal Whitening Toothpaste 75ml", "Oral care", "75 ml", "8710400100041", "8712000300043")]:
    P.append(product("Unilever", d, cat, size, go, gn, "2027-03-22", None, "sample_document"))
# Made-up PAST changeovers with a planted effect (what the impact analysis should recover)
for c, d, cat, size, bo, bn, lw, eff in [
    ("NIVEA", "NIVEA Creme Dose 75ml", "Hautpflege", "75 ml", "400580012005", "400638133505", "2026-03-02", 0.08),
    ("NIVEA", "NIVEA Sun Protect SPF30 200ml", "Sonnenschutz", "200 ml", "400580012006", "400638133506", "2026-04-13", 0.15),
    ("Henkel", "Persil Color Gel 1.5L", "Laundry", "1.5 L", "401500010005", "401580030005", "2026-05-04", -0.06),
    ("Henkel", "Somat Classic Tabs 40pcs", "Dishwashing", "40 pcs", "401500010006", "401580030006", "2026-06-01", 0.00),
    ("Unilever", "Dove Original Deo Spray 150ml", "Deodorant", "150 ml", "871040010005", "871200030005", "2026-07-06", 0.12),
    ("Unilever", "Axe Apollo Shower Gel 250ml", "Body care", "250 ml", "871040010006", "871200030006", "2026-08-03", -0.10)]:
    P.append(product(c, d, cat, size, gtin13(bo), gtin13(bn), lw, eff, "past_synthetic"))
# Controls: 3 never-changing products per category
CONTROLS = {
    "Hautpflege": ("NIVEA", ["NIVEA Soft Jar 200ml", "NIVEA Creme Tube 30ml", "NIVEA Q10 Day Cream 50ml"], "4005800121"),
    "Sonnenschutz": ("NIVEA", ["NIVEA Sun Kids SPF50 200ml", "NIVEA Sun After Sun 200ml", "NIVEA Sun Spray SPF30 200ml"], "4005800122"),
    "Rasur": ("NIVEA", ["NIVEA Men Shaving Foam 200ml", "NIVEA Men After Shave Balm 100ml", "NIVEA Men Shaving Gel 200ml"], "4005800123"),
    "Laundry": ("Henkel", ["Persil Discs 25pcs", "Persil Powder 1.3kg", "Perwoll Wool 1L"], "4015000101"),
    "Dishwashing": ("Henkel", ["Pril Lemon 500ml", "Somat Gold Tabs 36pcs", "Pril Balsam 500ml"], "4015000102"),
    "Hair care": ("Henkel", ["Gliss Conditioner 200ml", "Schauma Shampoo 400ml", "Taft Hairspray 250ml"], "4015000103"),
    "Body care": ("Unilever", ["Dove Body Wash 250ml", "Dove Shower Mousse 200ml", "Axe Black Shower Gel 400ml"], "8710400101"),
    "Deodorant": ("Unilever", ["Rexona Women Deo Spray 150ml", "Rexona Men Roll-on 50ml", "Dove Men Deo Spray 150ml"], "8710400102"),
    "Food": ("Unilever", ["Knorr Chicken Bouillon 8x10g", "Knorr Beef Bouillon 8x10g", "Hellmann's Mayonnaise 400ml"], "8710400103"),
    "Oral care": ("Unilever", ["Signal Family Toothpaste 75ml", "Signal Kids Toothpaste 50ml", "Signal White Now 75ml"], "8710400104"),
}
for cat, (client, names, prefix) in CONTROLS.items():
    for i, d in enumerate(names, 1):
        g = gtin13(f"{prefix}{i:02d}")
        P.append(product(client, d, cat, d.split()[-1], g, g, None, None, "control"))

assert all(len(p["gtin_old"]) == 13 and len(p["gtin_new"]) == 13 for p in P)
assert len({p["gtin_old"] for p in P}) == len(P), "duplicate GTINs"
print(len(P), "products:", pd.Series([p["role"] for p in P]).value_counts().to_dict())

In [0]:
promo_weeks = {w: stable("promo", w.date()) % 10 == 0 for w in WEEKS}  # ~10% of weeks are chain-wide promo weeks
rows = []
for p in P:
    rng = np.random.default_rng(stable("product", p["gtin_old"]) % 2**32)  # per-product seed: other products don't shift it
    price, base, trend = round(float(rng.uniform(1.49, 8.99)), 2), float(rng.uniform(40, 160)), float(rng.normal(0.0008, 0.0006))
    amp, peak = SEASON[p["category"]]
    for wi, wk in enumerate(WEEKS):
        woy = wk.isocalendar().week
        promo = promo_weeks[wk] and stable(p["desc"], wk.date()) % 3 == 0
        level = base * (1 + amp * np.cos(2 * np.pi * (woy - peak) / 52)) * (1.15 if woy >= 49 else 1.0) * (1 + trend) ** wi * (1.3 if promo else 1.0)
        shares = [(p["gtin_old"], 1.0)]
        if p["changeover"] is not None and wk >= p["changeover"] and p["gtin_old"] != p["gtin_new"]:
            k, eff = (wk - p["changeover"]).days // 7, 1 + (p["effect"] or 0)
            shares = [(p["gtin_old"], TRANSITION_OLD[k]), (p["gtin_new"], TRANSITION_NEW[k] * eff)] if k < 3 else [(p["gtin_new"], eff)]
        for store, scale in STORES.items():
            for g, share in shares:
                mu = level * scale * share
                units = int(rng.negative_binomial(NB_DISPERSION, NB_DISPERSION / (NB_DISPERSION + mu))) if mu > 0 else 0
                pr = round(price * (0.8 if promo else 1.0), 2)
                rows.append((wk.date(), store, g, p["client_id"], p["desc"], p["category"], units, pr, round(units * pr, 2), bool(promo)))
sales = pd.DataFrame(rows, columns=["week", "store_id", "gtin", "client_id", "article_description", "category", "units", "price_eur", "revenue_eur", "on_promo"])
print(len(sales), "rows |", sales.gtin.nunique(), "GTINs |", int(sales.units.sum()), "units")

In [0]:
# Difference-in-differences on the stitched old+new series: 12 weeks before vs weeks 4-15 after, against the category's controls
by = sales.assign(week=pd.to_datetime(sales.week)).groupby(["article_description", "week"])["units"].sum()
def ratio(desc, c):
    s = by.loc[desc]
    pre = s[(s.index >= c - pd.Timedelta(weeks=12)) & (s.index < c)].sum()
    post = s[(s.index >= c + pd.Timedelta(weeks=3)) & (s.index < c + pd.Timedelta(weeks=15))].sum()
    return post / pre
checks = []
for p in [p for p in P if p["role"] == "past_synthetic"]:
    controls = [q["desc"] for q in P if q["role"] == "control" and q["category"] == p["category"]]
    r_p, r_c = ratio(p["desc"], p["changeover"]), np.mean([ratio(d, p["changeover"]) for d in controls])
    checks.append(dict(product=p["desc"], changeover=p["changeover"].date(), planted=round(p["effect"] * 100, 1),
                       naive_before_after=round((r_p - 1) * 100, 1), estimate_vs_controls=round((r_p / r_c - 1) * 100, 1),
                       error_pts=round((r_p / r_c - 1 - p["effect"]) * 100, 1), controls=len(controls),
                       zero_weeks=int((by.loc[p["desc"]] == 0).sum())))
checks = pd.DataFrame(checks)
display(checks)
assert (checks.zero_weeks == 0).all(), "a changeover has a gap in sales"

In [0]:
now = pd.Timestamp.utcnow().tz_localize(None)
spark.createDataFrame(sales.assign(is_synthetic=True, generated_at=now)).write.mode("overwrite").option("overwriteSchema", "true") \
     .saveAsTable(f"{CATALOG}.silver.weekly_sales")
spark.sql(f"""COMMENT ON TABLE {CATALOG}.silver.weekly_sales IS
  'SYNTHETIC demo data: weekly units and revenue per GTIN and store (4 stores, 104 Monday-weeks). Generated by src/synthetic/generate_synthetic_sales; not real sales.'""")

truth = pd.DataFrame([dict(client_id=p["client_id"], article_description=p["desc"], category=p["category"], gtin_old=p["gtin_old"], gtin_new=p["gtin_new"],
                           changeover_week=p["changeover"].date() if p["changeover"] is not None else None,
                           planted_effect_pct=p["effect"] * 100 if p["effect"] is not None else None,
                           transition_weeks=3 if p["changeover"] is not None else None, product_role=p["role"], generated_at=now) for p in P])
spark.createDataFrame(truth).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.synthetic_changeover_truth")
spark.sql(f"""COMMENT ON TABLE {CATALOG}.silver.synthetic_changeover_truth IS
  'Answer key for the synthetic sales: planted sales effect per changeover and each product''s role. Use only to check the impact analysis.'""")

# Old -> new GTIN mapping for the past changeovers (inserted once; reruns don't duplicate)
pl = pd.DataFrame([dict(document_id=hashlib.md5(f"synthetic://seed/changeovers/{p['gtin_old']}".encode()).hexdigest(), client_id=p["client_id"],
                        launch_date=p["changeover"].date(), article_description=p["desc"], identifier_old=p["gtin_old"], identifier_new=p["gtin_new"],
                        identifier_type="GTIN", size=p["size"], packaging=None, category=p["category"],
                        notes=f"Synthetic past changeover for the sales-impact demo (planted effect {p['effect']:+.0%})", extracted_at=now,
                        source_file=f"synthetic://seed/changeovers/{p['client_id']}") for p in P if p["role"] == "past_synthetic"])
spark.createDataFrame(pl, schema=spark.table(f"{CATALOG}.silver.product_launches").schema).createOrReplaceTempView("new_changeovers")
spark.sql(f"""MERGE INTO {CATALOG}.silver.product_launches t USING new_changeovers s ON t.document_id = s.document_id
              WHEN NOT MATCHED THEN INSERT *""")

display(spark.sql(f"""
  SELECT (SELECT COUNT(*) FROM {CATALOG}.silver.weekly_sales) AS sales_rows,
         (SELECT COUNT(DISTINCT gtin) FROM {CATALOG}.silver.weekly_sales) AS gtins,
         (SELECT COUNT(*) FROM {CATALOG}.silver.synthetic_changeover_truth) AS products,
         (SELECT COUNT(*) FROM {CATALOG}.silver.product_launches WHERE source_file LIKE 'synthetic://seed/changeovers/%') AS past_changeovers"""))